In [5]:
import pandas as pd
import numpy as np

In [6]:
use_cols = [
    "item_id", "store_id", "date", "sales",
    "weekday", "wday", "month", "year",
    "event_name_1", "event_type_1"
]

chunks = pd.read_csv(
    "../data/interim/sales_long.csv",
    usecols=use_cols,
    parse_dates=["date"],
    chunksize=500_000,      
    engine="python"         
)


In [7]:
def create_lag_features(df):
    df = df.sort_values("date")
    
    # Lag features
    for lag in [7, 14, 28]:
        df[f"lag_{lag}"] = df["sales"].shift(lag)
    
    # Rolling features
    for window in [7, 14, 28]:
        df[f"rmean_{window}"] = (
            df["sales"]
            .shift(1)
            .rolling(window)
            .mean()
        )
        
    return df


In [8]:
processed_chunks = []

for chunk in chunks:
    for store_id, store_df in chunk.groupby("store_id"):
        
        store_df = (
            store_df
            .groupby("item_id", group_keys=False)
            .apply(create_lag_features,include_groups=False)
        )
        
        processed_chunks.append(store_df)


In [9]:
processed_chunks = [
    df.assign(is_event=df["event_name_1"].notna().astype(int))
    for df in processed_chunks
]


In [10]:
fe_df = pd.concat(processed_chunks, ignore_index=True)


In [11]:
# Drop rows with NaNs caused by lagging
fe_df = fe_df.dropna().reset_index(drop=True)


In [12]:
fe_df.to_csv(
    "../data/processed/train_fe.csv",
    index=False
)

## Feature Engineering Summary

- Created lag features (7, 14, 28 days) to capture temporal dependencies
- Created rolling mean features to smooth intermittent demand
- Added calendar features and event indicators
- Feature engineering was performed group-wise and chunk-wise to handle memory constraints
- Rows with insufficient historical data were removed

These features form the core inputs for downstream forecasting models.
